# Incident detection and recovery

**Module 18 — Security Logging, Monitoring and Incident Recovery**

Detection is a rule that fires. Recovery is a sequence of verified steps. This notebook replays a synthetic
incident against a mock service: credential stuffing, then quota abuse, then a data-scope probe. We measure time
to detect, contain, and recover — and check that each recovery step actually restored the property it claims to.

## Learning objectives

1. Write detection rules with explicit conditions and severities.
2. Build an incident timeline from events, and compute MTTD and MTTR.
3. Contain an incident with reversible, logged actions.
4. Verify recovery instead of assuming it.
5. Convert findings into controls with owners and tests.

## Background

The incident lifecycle this notebook follows:

**detect → triage → contain → eradicate → recover → verify → learn**

Two disciplines separate a real response from a scramble:

* **Containment is logged and reversible.** Revoking a key is fine; deleting data is not.
* **Recovery is verified.** "The service is up" is not the same as "the abused key no longer works and the
  affected records were identified".

All events are synthetic and the "attacker" is a random process inside this notebook. No external system is
involved.

## Dependencies and installation

```bash
python -m pip install "numpy>=2.1" "pandas>=2.2"
```

Standard library plus NumPy and pandas. No network, no credentials, no real users.

In [1]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

SEED = 73
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 160)
T0 = datetime(2026, 3, 4, 2, 0, 0)
print("ready")

ready


**Expected output** — `ready`.

### Step 1 — Synthetic events over 90 minutes

Normal traffic plus an attack that starts at minute 20. Each event has a time, principal, status and byte count,
which is what a real gateway log would carry.

In [2]:
events = []
for minute in range(90):
    for _ in range(int(rng.integers(4, 9))):
        events.append(
            {
                "t": T0 + timedelta(minutes=minute, seconds=int(rng.integers(0, 59))),
                "minute": minute,
                "principal": f"user-{int(rng.integers(1, 40)):02d}",
                "status": 200 if rng.random() > 0.08 else 401,
                "bytes": int(rng.integers(200, 900)),
                "route": "/predict",
            }
        )

for minute in range(20, 34):
    events.append({"t": T0 + timedelta(minutes=minute, seconds=5), "minute": minute,
                   "principal": "user-07", "status": 401, "bytes": 120, "route": "/predict"})
    events.append({"t": T0 + timedelta(minutes=minute, seconds=6), "minute": minute,
                   "principal": "user-07", "status": 401, "bytes": 120, "route": "/predict"})

for minute in range(34, 46):
    events.append({"t": T0 + timedelta(minutes=minute, seconds=10), "minute": minute,
                   "principal": "user-07", "status": 200, "bytes": 4200, "route": "/export"})

log = pd.DataFrame(events).sort_values("t").reset_index(drop=True)
print(len(log), "events; routes:", sorted(log.route.unique()))

594 events; routes: ['/export', '/predict']


**Expected output** — roughly 500 events over two routes, with the attack concentrated on `user-07`. The
`/export` route is the one the attacker escalates to, which is what the detection rules look for.

### Step 2 — Detection rules

Three rules, each with a condition, a severity and a reason string. Rules that cannot explain themselves cannot
be tuned.

In [3]:
def rule_credential_stuffing(df, threshold=8):
    fails = df[(df.status == 401)].groupby("principal").size()
    hits = fails[fails >= threshold]
    return [{"rule": "credential stuffing", "principal": p, "count": int(c), "severity": "high"} for p, c in hits.items()]


def rule_large_export(df, byte_threshold=3000, min_count=5):
    exports = df[(df.route == "/export") & (df.bytes > byte_threshold)]
    counts = exports.groupby("principal").size()
    return [{"rule": "bulk export", "principal": p, "count": int(c), "severity": "high"}
            for p, c in counts.items() if c >= min_count]


def rule_off_hours(df, start_hour=1, end_hour=5):
    hits = df[(df.t.dt.hour >= start_hour) & (df.t.dt.hour < end_hour)]
    counts = hits.groupby("principal").size()
    return [{"rule": "off-hours volume", "principal": p, "count": int(c), "severity": "medium"}
            for p, c in counts.items() if c >= 25]


alerts = rule_credential_stuffing(log) + rule_large_export(log)
print(f"{len(alerts)} alerts")
pd.DataFrame(alerts)

2 alerts


,rule,principal,count,severity
0,credential stuffing,user-07,31,high
1,bulk export,user-07,12,high


**Expected output** — at least two alerts, both for `user-07`: one for repeated 401s and one for bulk export.
The off-hours rule is deliberately not applied here; every event in this window is off-hours, which is the
false-positive trap a real deployment must handle with a per-user baseline.

### Step 3 — Timeline and time-to-detect

The incident timeline is built from events plus analyst actions. MTTD is measured from the first malicious event
to the alert, and MTTR from the alert to verified recovery.

In [4]:
first_bad = log[(log.principal == "user-07") & (log.status == 401) & (log.minute >= 20)].t.min()
alert_time = T0 + timedelta(minutes=46)          # analyst sees the export alert
timeline = pd.DataFrame(
    [
        {"time": first_bad, "actor": "attacker", "event": "first credential-stuffing attempt"},
        {"time": first_bad + timedelta(minutes=14), "actor": "attacker", "event": "first successful login"},
        {"time": first_bad + timedelta(minutes=14), "actor": "attacker", "event": "bulk export begins"},
        {"time": alert_time, "actor": "monitoring", "event": "bulk-export alert raised"},
        {"time": alert_time + timedelta(minutes=3), "actor": "analyst", "event": "triage: confirmed abuse"},
        {"time": alert_time + timedelta(minutes=6), "actor": "analyst", "event": "key revoked, account suspended"},
        {"time": alert_time + timedelta(minutes=20), "actor": "engineer", "event": "affected exports identified"},
        {"time": alert_time + timedelta(minutes=35), "actor": "engineer", "event": "recovery verified"},
    ]
)
timeline["elapsed_min"] = ((timeline.time - first_bad).dt.total_seconds() / 60).round(1)
mttd = (alert_time - first_bad).total_seconds() / 60
mttr = (timeline.time.iloc[-1] - alert_time).total_seconds() / 60
print(f"MTTD {mttd:.0f} min | MTTR {mttr:.0f} min")
timeline[["elapsed_min", "actor", "event"]]

MTTD 26 min | MTTR 35 min


,elapsed_min,actor,event
0,0.0,attacker,first credential-stuffing attempt
1,14.0,attacker,first successful login
2,14.0,attacker,bulk export begins
3,25.9,monitoring,bulk-export alert raised
4,28.9,analyst,triage: confirmed abuse
5,31.9,analyst,"key revoked, account suspended"
6,45.9,engineer,affected exports identified
7,60.9,engineer,recovery verified


**Expected output** — MTTD of 26 minutes and MTTR of 35 minutes, with an eight-row timeline. The 26 minutes is
the number to attack in the next quarter: detection was the slow part, not the response.

### Step 4 — Containment as logged, reversible actions

Each action has a state and a verification step. Reversible actions are preferred because a wrong containment
decision during triage is common.

In [5]:
actions = pd.DataFrame(
    [
        {"action": "suspend principal user-07", "reversible": True, "state": "done",
         "verify": "login attempt returns 403"},
        {"action": "revoke API key for user-07", "reversible": False, "state": "done",
         "verify": "old key returns 401"},
        {"action": "disable /export for the tenant", "reversible": True, "state": "done",
         "verify": "route returns 403 for all principals"},
        {"action": "rotate signing key", "reversible": False, "state": "pending",
         "verify": "tokens issued before rotation fail"},
    ]
)
actions

,action,reversible,state,verify
0,suspend principal user-07,True,done,login attempt returns 403
1,revoke API key for user-07,False,done,old key returns 401
2,disable /export for the tenant,True,done,route returns 403 for all principals
3,rotate signing key,False,pending,tokens issued before rotation fail


**Expected output** — four actions with reversibility, state and a verification sentence each. The pending
rotation is deliberate: it needs a change window, and the table makes that visible instead of leaving it as
folklore.

### Step 5 — Verify recovery with checks, not assertions of intent

Recovery verification is executable. Each check returns true or false, and the incident is not closed until all
are true.

In [6]:
def verify_recovery():
    checks = {
        "abused principal blocked": True,
        "old credential rejected": True,
        "export disabled for tenant": True,
        "affected records enumerated": True,
        "signing key rotated": False,
    }
    return checks


checks = verify_recovery()
frame = pd.DataFrame([{"check": k, "pass": v} for k, v in checks.items()])
print(f"recovery verified: {frame['pass'].all()}")
frame

recovery verified: False


,check,pass
0,abused principal blocked,True
1,old credential rejected,True
2,export disabled for tenant,True
3,affected records enumerated,True
4,signing key rotated,False


**Expected output** — `recovery verified: False` with four of five checks passing. The false row is the honest
state: the incident cannot be closed while a non-reversible containment step is outstanding. Change the rotation
to true only after it has actually happened.

### Step 6 — Lessons into controls

Every finding becomes a control with an owner and a test. A lesson without a test will be lost by the next
release.

In [7]:
lessons = pd.DataFrame(
    [
        {"finding": "401 burst took 26 min to notice", "control": "alert on failed-login rate per principal",
         "owner": "platform", "test": "replay this window in staging, expect one alert"},
        {"finding": "export route had no per-tenant cap", "control": "per-principal export quota",
         "owner": "api", "test": "automated test asserting 429 after N exports"},
        {"finding": "affected records needed manual enumeration",
         "control": "export audit table with principal and byte count", "owner": "data",
         "test": "query returns rows for the incident window"},
    ]
)
lessons

,finding,control,owner,test
0,401 burst took 26 min to notice,alert on failed-login rate per principal,platform,"replay this window in staging, expect one alert"
1,export route had no per-tenant cap,per-principal export quota,api,automated test asserting 429 after N exports
2,affected records needed manual enumeration,export audit table with principal and byte count,data,query returns rows for the incident window


**Expected output** — three findings, each with an owner and a test. The test column is what makes the table a
commitment rather than a summary.

## Exercises

1. **Improve detection.** Add a rule that alerts within five minutes of the 401 burst, and measure the
   false-positive rate on this log.
2. **Reconstruct the blast radius.** Using the event table, compute how many bytes were exported between the
   first success and containment.
3. **Break the timeline.** Assume the analyst was paged at minute 60 instead of 46. Recompute MTTD and MTTR and
   state what the delay cost in exported bytes.
4. **Write the customer notice.** Draft four sentences telling an affected tenant what happened, what was
   accessed, and what you have changed.

## Reflection questions

1. Why is MTTD usually the harder number to improve, and what does that imply about investment?
2. Which containment action in this notebook is irreversible, and what would you do instead if the key could not
   be rotated quickly?
3. Why must a recovery check be executable rather than a paragraph in a runbook?

## References

1. NIST, *SP 800-61 Rev. 2*, Computer Security Incident Handling Guide.
2. NIST, *AI 600-1*, Generative AI Profile, 2024 — incident response for AI systems.
3. OWASP, *ASVS* 4.0.3, V7 (logging) and V1 (architecture).
4. Cichonski, P. et al., *SP 800-61* — detection, containment, recovery phases.